## AKI Calculation - Description

This notebook is our first attempt to identify the subjects with AKI in our cohort, based on the KDIGO SCr criteria.

The code to calculate AKI within the VPS-PN cohort can be found here: https://github.com/Lab-for-Integrated-Decision-Support/vps-peds-mods. The relevant code and folders are:
The general process is as follows:

- Load the necessary data files, including ICU admissions, hospital admissions, laboratory results, and vial signs
- Select, clean, and join the data files as needed
  - Remove those with a LOS < 12 hours, as we are calculating risk at 12 hrs
  - Ensure the weight and height are within the proper distribution
  - Ensure the creatinines are within the proper distribution
  - Filter for age < 18 
- Compute the baseline SCr
  - Requires functions for height-dependent and height-independent from Hessey 2017 article
  - Also requires Schwartz min/max estimates for ensuring we are within an appropriate range
  - Use logic for computing based on lines ~ 360 - 375 in f_aki_functions.R script

- From that, compute the AKI stage for each SCr that resulted between 0 and 72 hrs of hospital admission
  - Use KDIGO SCr criteria, found on lines 430 - 435 of f_aki_functions.R file
  - If AKI > stage 0 (no AKI) before 12 hrs, then patient is excluded for having AKI on admisison
  - Otherwise, look for highest stage achieved between 12 and 72 hrs - this is our stage label for that patient
- Summarize AKI presence in cohort and mortality among those with AKI at 72 hrs
- Gather covariates - recall that for the original model, this includes "cardiac arrest pre-admission" which we will have to get from diagnoses codes, and "post-operative" which we likely cannot include.
- Calculate AKI risk based on the Sanchez-Pinto 2016 coefficients on the covariates
  - Generate an ROC curve based on this score and report the AUROC
  - Calculate the thresholds at 50% and 90% and report results

### Initialize and Load Functions

First we set the folder path, and define the function `load_data_file` to load the specific PDC data file from this folder.

In [3]:
import pandas as pd
import numpy
import os, sys, importlib 

try:
    import pyspark
    import pyspark.pandas as ps
except:
    %pip install pyspark
    import pyspark
    import pyspark.pandas as ps

try:
    import plotly.express as px
except:
    %pip install plotly
    import plotly.express as px
    
import azure_BlobStorageConnection

import load_utils

Set environment variables using script.

In [4]:
from set_env_vars import set_all_env_vars

set_all_env_vars()

Add global `debug_` variables to determine when to print out counts, show data tables, and plot graphs. This speeds processing in the PySpark environment, but may not matter if we're working solely with pandas data frames.

In [5]:
debug_count = False
debug_show = False
debug_plot = True

Initialize spark session.

In [6]:
spark = pyspark.sql.SparkSession.builder.appName('find_icu_aki').getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/anaconda/envs/azureml_py38/lib/python3.8/site-packages/pyspark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/anaconda/envs/azureml_py38/lib/python3.8/site-packages/pyspark/jars/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]


24/07/22 13:42:01 WARN Utils: Your hostname, sdrury-compute resolves to a loopback address: 127.0.0.1; using 10.0.0.4 instead (on interface eth0)
24/07/22 13:42:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/07/22 13:42:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/07/22 13:42:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Load pandas data frames of the four necessary data files.

In [7]:
# Load the data based on the imported functions above
# d_s = load_utils.getDataStore()

# h_df = load_utils.load_data_file('hospitaladmissions', d_s, path = '8-site-pandas/', debug = debug_count)
# h_lim = h_df[["pdc_hid", "pdc_pid", "sex", "ageatadmission", "hospitaldischarge", "hospitalmortality", "site_id"]]
# del(h_df)

# i_df = load_utils.load_data_file('icuadmissions', d_s, path = '8-site-pandas/', debug = debug_count)
# i_lim = i_df[["pdc_eid", "pdc_hid", "icumedicaldischarge", "icuadmission", "icumortality", "icudischarge", "site_id"]]
# del(i_df)

# labs_df = load_utils.load_data_file('labs', d_s, path = '8-site-pandas/', debug = debug_count)
# labs_lim = labs_df[["pdc_hid", "pdc_pid", "resulttime", "pdc_name", "source_units", "source_value"]]
# del(labs_df)

# vs_df = load_utils.load_data_file('vitalsigns', d_s, path = '8-site-pandas/', debug = debug_count)
# vs_lim = vs_df[["pdc_hid", "source_value", "pdc_name", "source_units", "starttime"]]

# del(vs_df)

In [8]:
joined_bscr = pd.read_csv(os.environ['AKI_DATA_PDC'] + 'joined_bscr.csv')
scr_join_left_icu = pd.read_csv(os.environ['AKI_DATA_PDC'] + 'scr_join_left_icu.csv')
base_cohort = pd.read_csv(os.environ['AKI_DATA_PDC'] + 'base_cohort_v2.csv')

/tmp/ipykernel_15740/856878314.py:2: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  scr_join_left_icu = pd.read_csv('../../scr_join_left_icu.csv')


# Calculate ICU AKI

Now that we have baseline AKI values for each row tuple `pdc_hid`, `pdc_eid` in the `base_cohort`, we need to calculate which of these rows had AKI on admission. We will then use similar code (changing the time frame) to calculate which had AKI between hours 12 and 72, and this will give us our outcome measure.

In [9]:
len(base_cohort)

138817

In [10]:
joined_bscr['bscr_source'].value_counts()

bscr_source
height_independent    81920
height_dependent      56890
Name: count, dtype: int64

In [11]:
icu_scr_staging = scr_join_left_icu[[
    'pdc_hid',
    'pdc_eid',
    'site_id',
    'n_icuadmit',
    'n_res_time',
    'val_numeric'
]].merge(
    joined_bscr,
    on=[
        'pdc_hid',
        'pdc_eid',
        'site_id'
    ],
    how='left'
)

icu_scr_staging['hrs_since_icu_admit'] = (icu_scr_staging['n_res_time'] - icu_scr_staging['n_icuadmit']) / (60. * 60.)

icu_scr_staging['scr_increase'] = icu_scr_staging['val_numeric'] - icu_scr_staging['bscr']
icu_scr_staging['fold_change'] = icu_scr_staging['scr_increase'] / icu_scr_staging['bscr']

def get_aki_stage_numeric(scr_row):
    if (scr_row['fold_change'] >= 3) or (scr_row['val_numeric'] >= 4):
        return 3
    elif (scr_row['fold_change'] >= 2) and (scr_row['fold_change'] < 3):
        return 2
    elif ((scr_row['fold_change'] >= 1.5) and (scr_row['fold_change'] < 2)) or ((scr_row['hrs_since_icu_admit'] / (60**2) <= 48) and (scr_row['scr_increase'] >= 0.3)):
        return 1
    else:
        return 0
    
icu_scr_staging['aki_stage_numeric'] = [get_aki_stage_numeric(icu_scr_staging.iloc[i]) for i in range(len(icu_scr_staging))]

In [12]:
if debug_count: print('Count of rows in icu_scr_staging:', len(icu_scr_staging))

if debug_show: icu_scr_staging.head()

Count of rows in icu_scr_staging: 622291


### Calculate AKI stages in windows

Now we calculate the maximum AKI stage in each of the relevant windows, including <12 hours from admission and 12-72 hours after admission.

In [13]:
aki_on_admit = icu_scr_staging[icu_scr_staging['hrs_since_icu_admit'] <= 12] \
.groupby(['pdc_hid', 'pdc_eid']) \
.max() \
.reset_index() \
.rename(columns={'aki_stage_numeric': 'max_aki_admit'})[[
    'pdc_hid',
    'pdc_eid',
    'max_aki_admit'
]]

if debug_count: print('# of rows in aki_on_admit dataframe:', len(aki_on_admit))
if debug_show: aki_on_admit.head()

# of rows in aki_on_admit dataframe: 66665


In [14]:
aki_12_to_72 = icu_scr_staging[(icu_scr_staging['hrs_since_icu_admit'] > 12) & (icu_scr_staging['hrs_since_icu_admit'] <= 72)] \
.groupby(['pdc_hid', 'pdc_eid']) \
.max() \
.reset_index() \
.rename(columns={'aki_stage_numeric': 'max_aki_stay'})[[
    'pdc_hid',
    'pdc_eid',
    'max_aki_stay'
]]

if debug_count: print('# of rows in aki_12_to_72 dataframe:', len(aki_12_to_72))
if debug_show: aki_12_to_72.head()

# of rows in aki_12_to_72 dataframe: 68208


In [15]:
final_aki_cohort = base_cohort[[
    'ageatadmission',
    'pdc_hid',
    'pdc_eid',
    'site_id',
    'icumortality',
    'pdc_pid',
    'n_icuadmit'
]].copy()

final_aki_cohort['age_admit_yrs'] = final_aki_cohort['ageatadmission'].astype(float) / 365.25
final_aki_cohort.drop('ageatadmission', axis=1, inplace=True)

final_aki_cohort = final_aki_cohort[final_aki_cohort['age_admit_yrs'] <= 18]

final_aki_cohort = final_aki_cohort \
.merge(
    aki_on_admit,
    on=['pdc_hid', 'pdc_eid'],
    how='left'
) \
.merge(
    aki_12_to_72,
    on=['pdc_hid', 'pdc_eid'],
    how='left'
)

final_aki_cohort['new_aki'] = numpy.where(
    final_aki_cohort['max_aki_admit'] > 0,
    'On Admit',
    numpy.where(
        final_aki_cohort['max_aki_stay'] > 0,
        'AKI',
        'No AKI'
    )
)

Here we group the final AKI cohort results by whether or not they have new AKI, and by mortality. These numbers are used in the AKI conference abstract.

In [16]:
if debug_count:
    
    print(
            final_aki_cohort[[
            'new_aki', 
            'icumortality',
            'pdc_eid'
        ]] \
        .groupby(['new_aki', 'icumortality']) \
        .count() \
        .reset_index() \
        .rename(columns={'pdc_eid': 'count'})
    )

    new_aki  icumortality   count
0       AKI           0.0    3159
1       AKI           1.0     376
2    No AKI           0.0  125425
3    No AKI           1.0    1626
4  On Admit           0.0    7415
5  On Admit           1.0     655


Export final_aki_cohort to CSV to use in later notebooks.

In [17]:
print('# of unique patients: %d' % final_aki_cohort['pdc_pid'].nunique())

# of unique patients: 90730


In [21]:
final_aki_cohort['max_aki_admit'] = final_aki_cohort['max_aki_admit'].fillna(0)

final_aki_cohort[final_aki_cohort['max_aki_admit']==0].to_csv(os.environ['AKI_DATA_PDC'] + 'final_aki_cohort.csv', index=False)

In [30]:
final_aki_cohort[
    final_aki_cohort['max_aki_admit']==0
]['max_aki_stay'].fillna(0).value_counts()

max_aki_stay
0.0    127167
1.0      2927
3.0       410
2.0       220
Name: count, dtype: int64